# installing

In [ ]:
!pip install transformers accelerate bitsandbytes xformers gradio


# importing

In [ ]:
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
    import torch
except Exception :
    !pip uninstall -y torch torchvision torchaudio
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 #require restart , after restarting run the next cell

In [ ]:
import os
import requests
from datetime import datetime
import gradio as gr
import re

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

In [ ]:
from huggingface_hub import login
login(token="")  # Replace with your Hugging Face API token

# steps

## OCR.space

In [ ]:
#lang = input ('enter image language ("ara", "eng")')

In [ ]:

import requests

def ocr_space(image_path, language):
    API_KEY = ""  # Replace with your OCR.space API key
    url = "https://api.ocr.space/parse/image"
    payload = {
        "language": language,
        "isTable": False,
        "OCREngine": 1
    }
    headers = {
        "apikey": API_KEY
    }

    with open(image_path, "rb") as f:
        response = requests.post(
            url,
            files={"file": f},
            data=payload,
            headers=headers
        )

    try:
        result = response.json()
        if "ParsedResults" in result:
            return result["ParsedResults"][0]["ParsedText"]
        else:
            print("API Response (No ParsedResults):", result)
            return None
    except Exception as e:
        print("Error decoding JSON:", e)
        print("Raw response:", response.text)
        return None


#img_path = "/content/drive/MyDrive/DEPI_Project_Dataset/Screenshot 2025-05-10 162935.png"
#
#
#text = ocr_space(img_path, language=lang)
#print(text)


## simple pre-processing

In [ ]:

def clean_ocr_text(text):
    # Remove BOM and other invisible chars
    text = text.replace('\ufeff', '')

    # Join broken abbreviations like "e.\ng."
    text = re.sub(r'e\.\s*\n\s*g\.', 'e.g.', text, flags=re.IGNORECASE)

    # Fix cases like ":.", "..", ". .", etc.
    text = re.sub(r':\s*\.+', ':', text)
    text = re.sub(r'(\.\s*){2,}', '. ', text)
    text = re.sub(r'\s*\.\s*', '. ', text)
    text = re.sub(r'\s*,\s*', ', ', text)

    # Fix extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Restore line breaks after full stops if sentence is long enough
    text = re.sub(r'(?<=[a-z])\. (?=[A-Z])', '.\n', text)

    return text.strip()


In [ ]:
#text = clean_ocr_text(text)
#text

##model loading

In [ ]:

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


## Model functions

In [ ]:
def process_text_with_llama_summary(text: str, generator=generator) -> str:

    prompt = (f"""You are an expert summarizer. Create concise summaries that:
            1. Capture all key information
            2. Remove redundant details
            3. Maintain original meaning
            4. Are 3-5 sentences maximum
            5. The response language must be the same as the input text. For example, your response should be in Arabic if the input text is Arabic. also, if the input text is in English, your response should be in English
            6. Don't write anything except the summary
            7. Ensure that no thing from this prompt is in the response
            8. Ensure that you DONT TYPE ANY NOTES OR ADDONS TO THE RESPONSE
            9. JUST THE SUMMARIZED ANSWER
            Text to summarize:\n\n{text}\n\nProvide a professional summary
            """
            )

    output = generator(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)[0]["generated_text"]
    return output[len(prompt):].strip()


In [ ]:
def process_text_with_llama_Questions_and_Answers(text: str, generator=generator) -> str:
    prompt = (
        f"""You are a helpful assistant that generates 5 high-quality comprehension questions AND their answers based ONLY on the following technical text.
- The questions and answers must be in the same language as the input text.
- If the answer cannot be found explicitly, provide a concise answer based on the text.
- Do NOT add extra notes or explanations beyond the question and its answer.
- DO NOT ADD ANY ADDONS JUST THE QUESTIONS

Text:
{text}

Please provide the questions and answers now."""
    )
    output = generator(prompt, max_new_tokens=350, do_sample=True, temperature=0.7)[0]["generated_text"]
    return output[len(prompt):].strip()


## front-end

In [ ]:

def process(image, lang, function):

    if image is None or not lang or not function :
        return "Please upload an image and provide a language and select a function."

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    image_path = f"/content/{timestamp}.png"
    image.save(image_path)

    if lang == 'Arabic':
        lang = 'ara'
    elif lang == 'English':
        lang = 'eng'

    text = ocr_space(image_path, language=lang)
    text = clean_ocr_text(text)
    if function == 'summarize':
        text = process_text_with_llama_summary(text)
    elif function == 'generate questions':
        text = process_text_with_llama_Questioning(text)

    os.remove(image_path)

    return text



with gr.Blocks(css="""
    .small-button {
        max-width: 100px !important;
        min-width: 50px !important;
        margin: auto !important;
    }
""") as demo:
    with gr.Row():
        with gr.Column():
            img = gr.Image(type="pil", label="Upload Image")
            lang = gr.Radio(choices=['English', 'Arabic'], label="Image Language?", interactive=True)
            function = gr.Radio(
                choices=['summarize', 'generate questions'],
                label="Select Function",
                interactive=True
            )

        with gr.Column():
            output = gr.TextArea(label="Output",interactive=False, placeholder='Your result will appear here...' , show_copy_button=True)
    with gr.Row():
        submit = gr.Button("Submit", elem_classes="small-button")
    submit.click(fn=process, inputs=[img, lang, function], outputs=output)
demo.launch()
